In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [3]:
from app.ml.dataset_updater import update_dataset

result = update_dataset()

print(result)

{'status': 'success', 'message': 'Dataset berhasil diperiksa.', 'added': 0, 'dataset_size': 3938}


In [4]:
from app.ml.trainer import train_model

result = train_model()

print(result)

{'accuracy': 0.9631979695431472, 'dataset_size': 3938}


In [5]:
import os

print(os.path.getmtime("../models/model.pkl"))
print(os.path.getmtime("../models/vectorizer.pkl"))
print(os.path.getmtime("../models/label_encoder.pkl"))

1786461183.4755163
1786461183.5095184
1786461183.5095184


In [6]:
import joblib

svm = joblib.load("../models/model.pkl")
vectorizer = joblib.load("../models/vectorizer.pkl")
encoder = joblib.load("../models/label_encoder.pkl")

In [7]:
test_data = [
    "Beli nasi goreng",
    "Bayar listrik bulanan",
    "Top up GoPay",
    "Beli sepatu online",
    "Naik Grab ke kampus",
    "Bayar uang sekolah adik",
    "Beli kopi di kedai"
]

for text in test_data:

    vector = vectorizer.transform([text])

    pred = svm.predict(vector)

    kategori = encoder.inverse_transform(pred)

    print(f"{text} --> {kategori[0]}")

Beli nasi goreng --> food
Bayar listrik bulanan --> bills
Top up GoPay --> topup
Beli sepatu online --> shopping
Naik Grab ke kampus --> transport
Bayar uang sekolah adik --> education
Beli kopi di kedai --> food


In [8]:
from app.ml.dataset_updater import update_dataset

result = update_dataset()

print(result)

{'status': 'success', 'message': 'Dataset berhasil diperiksa.', 'added': 0, 'dataset_size': 3938}


In [9]:
from app.ml.retrainer import retrain_model

result = retrain_model()

print(result)

Accuracy model baru: 0.9632
{'status': 'success', 'accuracy': 0.9631979695431472, 'dataset_size': 3938, 'model': LinearSVC(), 'vectorizer': TfidfVectorizer(), 'encoder': LabelEncoder()}


In [10]:
import joblib

svm = joblib.load("../models/model.pkl")
vectorizer = joblib.load("../models/vectorizer.pkl")
encoder = joblib.load("../models/label_encoder.pkl")

In [11]:
test_data = [
    "Bayar uang sekolah adik",
    "Beli kopi di kedai",
    "Biaya konsultasi dokter",
    "Bayar listrik bulanan",
    "Naik Grab ke kampus"
]

for text in test_data:

    vector = vectorizer.transform([text])

    pred = svm.predict(vector)

    label = encoder.inverse_transform(pred)

    print(f"{text} -> {label[0]}")

Bayar uang sekolah adik -> education
Beli kopi di kedai -> food
Biaya konsultasi dokter -> healthcare
Bayar listrik bulanan -> bills
Naik Grab ke kampus -> transport


In [12]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

dataset_path = "../data/final/merged_dataset.csv"

df = pd.read_csv(dataset_path)
df = df.dropna(subset=["text", "label"])

X = df["text"].astype(str)
y = df["label"].astype(str)

print("Dataset:", len(df))

Dataset: 3938


In [13]:
old_model = joblib.load("../models/model.pkl")
old_vectorizer = joblib.load("../models/vectorizer.pkl")
old_encoder = joblib.load("../models/label_encoder.pkl")

print("Model lama berhasil dimuat.")

Model lama berhasil dimuat.


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
X_test_old = old_vectorizer.transform(X_test)

In [16]:
y_test_encoded = old_encoder.transform(y_test)

In [17]:
old_pred = old_model.predict(X_test_old)

old_accuracy = accuracy_score(
    y_test_encoded,
    old_pred
)

print(f"Accuracy model lama: {old_accuracy:.4f}")
print(f"Accuracy model lama: {old_accuracy * 100:.2f}%")

Accuracy model lama: 0.9632
Accuracy model lama: 96.32%


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC

In [19]:
new_encoder = LabelEncoder()

y_encoded = new_encoder.fit_transform(y)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [21]:
new_vectorizer = TfidfVectorizer()

X_train_tfidf = new_vectorizer.fit_transform(X_train)
X_test_tfidf = new_vectorizer.transform(X_test)

In [22]:
new_model = LinearSVC()

new_model.fit(
    X_train_tfidf,
    y_train
)

LinearSVC()

In [23]:
new_pred = new_model.predict(X_test_tfidf)

new_accuracy = accuracy_score(
    y_test,
    new_pred
)

print(f"Accuracy model baru: {new_accuracy:.4f}")
print(f"Accuracy model baru: {new_accuracy * 100:.2f}%")

Accuracy model baru: 0.9632
Accuracy model baru: 96.32%


In [24]:
print("================================")
print("MODEL COMPARISON")
print("================================")

print(f"Model lama : {old_accuracy * 100:.2f}%")
print(f"Model baru : {new_accuracy * 100:.2f}%")

if new_accuracy >= old_accuracy:
    print("KEPUTUSAN: MODEL BARU LEBIH BAIK / SAMA")
else:
    print("KEPUTUSAN: MODEL LAMA DIPERTAHANKAN")

MODEL COMPARISON
Model lama : 96.32%
Model baru : 96.32%
KEPUTUSAN: MODEL BARU LEBIH BAIK / SAMA


In [25]:
joblib.dump(new_model, "../models/model.pkl")
joblib.dump(new_vectorizer, "../models/vectorizer.pkl")
joblib.dump(new_encoder, "../models/label_encoder.pkl")

print("Model baru disimpan.")

Model baru disimpan.


In [26]:
from app.ml.retrainer import retrain_model

result = retrain_model()

print("Status:", result["status"])
print("Accuracy:", result["accuracy"])
print("Dataset:", result["dataset_size"])

Accuracy model baru: 0.9632
Status: success
Accuracy: 0.9631979695431472
Dataset: 3938


In [27]:
import os

model_path = "../models/model.pkl"

timestamp_sebelum = os.path.getmtime(model_path)

print("Timestamp SEBELUM retrain:")
print(timestamp_sebelum)

Timestamp SEBELUM retrain:
1786461185.647363


In [28]:
from app.ml.retrainer import retrain_model

result = retrain_model()

print("Accuracy model baru:", result["accuracy"])

Accuracy model baru: 0.9632
Accuracy model baru: 0.9631979695431472


In [29]:
timestamp_sesudah = os.path.getmtime(model_path)

print("Timestamp SESUDAH retrain:")
print(timestamp_sesudah)

Timestamp SESUDAH retrain:
1786461185.647363


In [30]:
print("Model lama tidak berubah:",
      timestamp_sebelum == timestamp_sesudah)

Model lama tidak berubah: True


In [31]:
import app.ml.retrainer

print(app.ml.retrainer.__file__)

d:\sarimiisi2\finsight\apps\ai-service\app\ml\retrainer.py


In [32]:
import importlib
import app.ml.retrainer

importlib.reload(app.ml.retrainer)

<module 'app.ml.retrainer' from 'd:\\sarimiisi2\\finsight\\apps\\ai-service\\app\\ml\\retrainer.py'>

In [33]:
from app.ml.retrainer import retrain_model

In [34]:
import os

model_path = "../models/model.pkl"

timestamp_sebelum = os.path.getmtime(model_path)

print("SEBELUM:", timestamp_sebelum)

SEBELUM: 1786461185.647363


In [35]:
result = retrain_model()

print("Accuracy:", result["accuracy"])

Accuracy model baru: 0.9632
Accuracy: 0.9631979695431472


In [36]:
timestamp_sesudah = os.path.getmtime(model_path)

print("SESUDAH:", timestamp_sesudah)

print(
    "Model berubah:",
    timestamp_sebelum != timestamp_sesudah
)

SESUDAH: 1786461185.647363
Model berubah: False


In [37]:
from app.ml.retrainer import retrain_model

result = retrain_model()

print("Accuracy model baru:", result["accuracy"])
print("Dataset:", result["dataset_size"])

Accuracy model baru: 0.9632
Accuracy model baru: 0.9631979695431472
Dataset: 3938


In [38]:
new_model = result["model"]
new_vectorizer = result["vectorizer"]
new_encoder = result["encoder"]

print("Model baru:", type(new_model))
print("Vectorizer baru:", type(new_vectorizer))
print("Encoder baru:", type(new_encoder))

Model baru: <class 'sklearn.svm._classes.LinearSVC'>
Vectorizer baru: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
Encoder baru: <class 'sklearn.preprocessing._label.LabelEncoder'>


In [39]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [40]:
old_model = joblib.load("../models/model.pkl")
old_vectorizer = joblib.load("../models/vectorizer.pkl")
old_encoder = joblib.load("../models/label_encoder.pkl")

print("Model lama berhasil dimuat.")

Model lama berhasil dimuat.


In [41]:
df = pd.read_csv("../data/final/merged_dataset.csv")

df = df.dropna(subset=["text", "label"])

X = df["text"].astype(str)
y = df["label"].astype(str)

print("Dataset:", len(df))

Dataset: 3938


In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [43]:
y_test_encoded = old_encoder.transform(y_test)

In [44]:
X_test_old = old_vectorizer.transform(X_test)

old_pred = old_model.predict(X_test_old)

old_accuracy = accuracy_score(
    y_test_encoded,
    old_pred
)

print(f"Accuracy model lama: {old_accuracy:.4f}")

Accuracy model lama: 0.9632


In [45]:
y_test_new_encoded = new_encoder.transform(y_test)

X_test_new = new_vectorizer.transform(X_test)

new_pred = new_model.predict(X_test_new)

new_accuracy = accuracy_score(
    y_test_new_encoded,
    new_pred
)

print(f"Accuracy model baru: {new_accuracy:.4f}")

Accuracy model baru: 0.9632


In [46]:
print("================================")
print("PERBANDINGAN MODEL")
print("================================")

print(f"Model lama : {old_accuracy:.4f}")
print(f"Model baru : {new_accuracy:.4f}")

if new_accuracy > old_accuracy:
    print("MODEL BARU LEBIH BAGUS")
elif new_accuracy == old_accuracy:
    print("ACCURACY SAMA")
else:
    print("MODEL BARU LEBIH JELEK")

PERBANDINGAN MODEL
Model lama : 0.9632
Model baru : 0.9632
ACCURACY SAMA


In [47]:
from app.ml.model_manager import compare_accuracy

In [48]:
comparison = compare_accuracy(
    old_accuracy,
    new_accuracy
)

print(comparison)

{'update': False, 'message': 'Model baru tidak lebih bagus. Model lama dipertahankan.'}


In [49]:
import joblib

from app.ml.retrainer import retrain_model


MODEL_PATH = "../models/model.pkl"
VECTORIZER_PATH = "../models/vectorizer.pkl"
ENCODER_PATH = "../models/label_encoder.pkl"


def save_model(model, vectorizer, encoder):

    joblib.dump(model, MODEL_PATH)
    joblib.dump(vectorizer, VECTORIZER_PATH)
    joblib.dump(encoder, ENCODER_PATH)

    return {
        "status": "success",
        "message": "Model baru berhasil disimpan."
    }


def compare_accuracy(old_accuracy, new_accuracy):

    if new_accuracy > old_accuracy:
        return {
            "update": True,
            "message": "Model baru lebih bagus."
        }

    return {
        "update": False,
        "message": "Model baru tidak lebih bagus. Model lama dipertahankan."
    }

In [50]:
from app.ml.evaluator import evaluate_model

In [51]:
old_accuracy = evaluate_model(
    old_model,
    old_vectorizer,
    old_encoder
)

print(
    f"Accuracy model lama: {old_accuracy:.4f}"
)

Accuracy model lama: 0.9632


In [52]:
new_accuracy = evaluate_model(
    new_model,
    new_vectorizer,
    new_encoder
)

print(
    f"Accuracy model baru: {new_accuracy:.4f}"
)

Accuracy model baru: 0.9632


In [53]:
comparison = compare_accuracy(
    old_accuracy,
    new_accuracy
)

print(comparison)

{'update': False, 'message': 'Model baru tidak lebih bagus. Model lama dipertahankan.'}


In [54]:
from app.ml.model_manager import retrain_and_update

In [55]:
result = retrain_and_update()

print(result)

Accuracy model baru: 0.9632
{'status': 'kept_old', 'old_accuracy': 0.9631979695431472, 'new_accuracy': 0.9631979695431472, 'message': 'Model lama dipertahankan.'}


In [56]:
import os

print(
    os.path.getmtime("../models/model.pkl")
)

1786461185.647363


In [57]:
from app.ml.model_manager import retrain_pipeline

In [58]:
result = retrain_pipeline()

print(result)

Accuracy model baru: 0.9632
{'status': 'kept_old', 'old_accuracy': 0.9631979695431472, 'new_accuracy': 0.9631979695431472, 'message': 'Model lama dipertahankan.'}


In [59]:
from app.ml.model_manager import compare_accuracy
print(
    compare_accuracy(
        0.95,
        0.96
    )
)

{'update': True, 'message': 'Model baru lebih bagus.'}


In [60]:
print(
    compare_accuracy(
        0.96,
        0.95
    )
)

{'update': False, 'message': 'Model baru tidak lebih bagus. Model lama dipertahankan.'}


In [61]:
print(
    compare_accuracy(
        0.96,
        0.96
    )
)

{'update': False, 'message': 'Model baru tidak lebih bagus. Model lama dipertahankan.'}


In [62]:
from app.ml.dataset_updater import update_dataset

result = update_dataset()

print(result)

{'status': 'success', 'message': 'Dataset berhasil diperiksa.', 'added': 0, 'dataset_size': 3938}


In [63]:
from app.ml.model_manager import retrain_pipeline

result = retrain_pipeline()

print(result)

Accuracy model baru: 0.9632
{'status': 'kept_old', 'old_accuracy': 0.9631979695431472, 'new_accuracy': 0.9631979695431472, 'message': 'Model lama dipertahankan.'}


In [64]:
from app.ml.dataset_updater import update_dataset
from app.ml.model_manager import retrain_pipeline

In [65]:
update_result = update_dataset()

print("UPDATE DATASET")
print(update_result)

UPDATE DATASET
{'status': 'success', 'message': 'Dataset berhasil diperiksa.', 'added': 0, 'dataset_size': 3938}


In [66]:
retrain_result = retrain_pipeline()

print("\nRETRAIN PIPELINE")
print(retrain_result)

Accuracy model baru: 0.9632

RETRAIN PIPELINE
{'status': 'kept_old', 'old_accuracy': 0.9631979695431472, 'new_accuracy': 0.9631979695431472, 'message': 'Model lama dipertahankan.'}


In [67]:
import os

old_timestamp = os.path.getmtime(
    "../models/model.pkl"
)

print("Timestamp:", old_timestamp)

Timestamp: 1786461185.647363


In [68]:
new_timestamp = os.path.getmtime(
    "../models/model.pkl"
)

print("Timestamp:", new_timestamp)

Timestamp: 1786461185.647363


In [69]:
from app.ml.dataset_updater import update_dataset

result = update_dataset()

print(result)

{'status': 'success', 'message': 'Dataset berhasil diperiksa.', 'added': 0, 'dataset_size': 3938}


In [70]:
from app.ml.retrainer import retrain_model

result = retrain_model()

print(
    "Accuracy:",
    result["accuracy"]
)

print(
    "Dataset:",
    result["dataset_size"]
)

Accuracy model baru: 0.9632
Accuracy: 0.9631979695431472
Dataset: 3938


In [71]:
import os

print(
    os.path.getmtime(
        "../models/model.pkl"
    )
)

1786461185.647363


In [72]:
from app.ml.model_manager import retrain_pipeline

result = retrain_pipeline()

print(result)

Accuracy model baru: 0.9632
{'status': 'kept_old', 'old_accuracy': 0.9631979695431472, 'new_accuracy': 0.9631979695431472, 'message': 'Model lama dipertahankan.'}


In [73]:
from app.ml.model_manager import retrain_pipeline

result = retrain_pipeline()

print(result)

Accuracy model baru: 0.9632
{'status': 'kept_old', 'old_accuracy': 0.9631979695431472, 'new_accuracy': 0.9631979695431472, 'message': 'Model lama dipertahankan.'}


In [74]:
import os

before = os.path.getmtime(
    "../models/model.pkl"
)

print("Before:", before)

Before: 1786461185.647363


In [75]:
after = os.path.getmtime(
    "../models/model.pkl"
)

print("After:", after)

After: 1786461185.647363


In [76]:
import pandas as pd

feedback_df = pd.read_csv(
    "../data/final/feedback_dataset.csv"
)

merged_df = pd.read_csv(
    "../data/final/merged_dataset.csv"
)

existing_texts = set(
    merged_df["text"]
    .astype(str)
    .str.strip()
)

pending = feedback_df[
    ~feedback_df["text"]
    .astype(str)
    .str.strip()
    .isin(existing_texts)
]

print("Feedback total :", len(feedback_df))
print("Pending        :", len(pending))

display(pending)

Feedback total : 45
Pending        : 2


,text,predicted_label,corrected_label,is_correct
43,nonton 510 di jakarta,travel,entertaiment,False
44,nonton itzy di jakarta,travel,entertaiment,False


In [77]:
import pandas as pd

feedback_df = pd.read_csv(
    "../data/final/feedback_dataset.csv"
)

merged_df = pd.read_csv(
    "../data/final/merged_dataset.csv"
)

existing_texts = set(
    merged_df["text"]
    .astype(str)
    .str.strip()
)

pending = feedback_df[
    (
        feedback_df["predicted_label"]
        != feedback_df["corrected_label"]
    )
    &
    (
        ~feedback_df["text"]
        .astype(str)
        .str.strip()
        .isin(existing_texts)
    )
].drop_duplicates(
    subset=["text"],
    keep="last"
)

print("Feedback total :", len(feedback_df))
print("Pending        :", len(pending))
print("Threshold      : 10")

display(pending)

Feedback total : 42
Pending        : 4
Threshold      : 10


,text,predicted_label,corrected_label,is_correct
38,nonton 510 di jakarta,travel,entertaiment,False
39,nonton itzy di jakarta,travel,entertaiment,False
40,Bayar tiket bus ke Bandung,shopping,travel,False
41,Bayar listrik rumah,shopping,bills,False


In [78]:
import pandas as pd

df = pd.read_csv(
    "../data/final/merged_dataset.csv"
)

print(
    sorted(
        df["label"]
        .dropna()
        .unique()
    )
)

['bills', 'donation', 'education', 'entertainment', 'fees', 'food', 'healthcare', 'income', 'investment', 'loan', 'shopping', 'topup', 'transfer', 'transport', 'travel']


In [84]:
import pandas as pd

df = pd.read_csv(
    "../data/final/merged_dataset.csv"
)

print("Dataset sekarang:", len(df))

Dataset sekarang: 3948


In [85]:
feedback_df = pd.read_csv(
    "../data/final/feedback_dataset.csv"
)

merged_df = pd.read_csv(
    "../data/final/merged_dataset.csv"
)

existing_texts = set(
    merged_df["text"]
    .astype(str)
    .str.strip()
)

pending = feedback_df[
    (
        feedback_df["predicted_label"]
        != feedback_df["corrected_label"]
    )
    &
    (
        ~feedback_df["text"]
        .astype(str)
        .str.strip()
        .isin(existing_texts)
    )
].drop_duplicates(
    subset=["text"],
    keep="last"
)

print("Feedback total :", len(feedback_df))
print("Pending        :", len(pending))

Feedback total : 50
Pending        : 1


In [86]:
import pandas as pd

feedback_df = pd.read_csv(
    "../data/final/feedback_dataset.csv"
)

merged_df = pd.read_csv(
    "../data/final/merged_dataset.csv"
)

feedback_df.columns = feedback_df.columns.str.strip()
merged_df.columns = merged_df.columns.str.strip()

existing_texts = set(
    merged_df["text"]
    .astype(str)
    .str.strip()
)

pending = feedback_df[
    (
        feedback_df["predicted_label"]
        != feedback_df["corrected_label"]
    )
    &
    (
        ~feedback_df["text"]
        .astype(str)
        .str.strip()
        .isin(existing_texts)
    )
].copy()

pending = pending.drop_duplicates(
    subset=["text"],
    keep="last"
)

print("Pending:", len(pending))

display(pending)

Pending: 1


,text,predicted_label,corrected_label,is_correct
48,Beli merch extrememerch,entertaiment,shopping,False


In [87]:
pending_text = pending.iloc[0]["text"]

print("TEXT:", pending_text)

print("\nFEEDBACK:")
display(
    feedback_df[
        feedback_df["text"].astype(str).str.strip()
        == pending_text.strip()
    ]
)

print("\nDATASET:")
display(
    merged_df[
        merged_df["text"].astype(str).str.strip()
        == pending_text.strip()
    ]
)

TEXT: Beli merch extrememerch

FEEDBACK:


,text,predicted_label,corrected_label,is_correct
48,Beli merch extrememerch,entertaiment,shopping,False



DATASET:


,text,label


In [88]:
from app.ml.dataset_updater import update_dataset

result = update_dataset()

print(result)

{'status': 'success', 'message': 'Dataset berhasil diperiksa.', 'added': 1, 'dataset_size': 3949}


In [89]:
import pandas as pd

merged_df = pd.read_csv(
    "../data/final/merged_dataset.csv"
)

result = merged_df[
    merged_df["text"]
    .astype(str)
    .str.strip()
    == "Beli merch extrememerch"
]

display(result)

,text,label
3948,Beli merch extrememerch,shopping


In [90]:
feedback_df = pd.read_csv(
    "../data/final/feedback_dataset.csv"
)

merged_df = pd.read_csv(
    "../data/final/merged_dataset.csv"
)

existing_texts = set(
    merged_df["text"]
    .astype(str)
    .str.strip()
)

pending = feedback_df[
    (
        feedback_df["predicted_label"]
        != feedback_df["corrected_label"]
    )
    &
    (
        ~feedback_df["text"]
        .astype(str)
        .str.strip()
        .isin(existing_texts)
    )
].drop_duplicates(
    subset=["text"],
    keep="last"
)

print("Pending:", len(pending))

display(pending)

Pending: 0


,text,predicted_label,corrected_label,is_correct


In [91]:
import pandas as pd

df = pd.read_csv(
    "../data/final/merged_dataset.csv"
)

print("Dataset sekarang:", len(df))

Dataset sekarang: 3949


In [92]:
import os

print(
    "Model timestamp:",
    os.path.getmtime("../models/model.pkl")
)

Model timestamp: 1786461185.647363
